# Union-Find (Disjoint Set Union)

Tracks a collection of non-overlapping sets. Supports two operations:

| Operation | Description | Time (optimized) |
|-----------|-------------|------------------|
| find(x) | Which set does x belong to? | O(α(n)) ≈ O(1) |
| union(x, y) | Merge the sets containing x and y | O(α(n)) ≈ O(1) |

α(n) = inverse Ackermann function -- grows so slowly it's effectively constant.

**Applications:** Kruskal's MST, cycle detection in undirected graphs, connected components, network connectivity.

## Key Optimizations

1. **Union by rank** -- attach the shorter tree under the taller one → keeps trees flat
2. **Path compression** -- during `find`, point every node directly to the root → flattens on access

Without optimizations: O(n) per operation. With both: O(α(n)) amortized.

> **Procedural style:** The data structure is just two arrays (`parent` and `rank`). Functions operate on them directly -- no wrapper class needed.

In [ ]:
import unittest

class UnionFindTests(unittest.TestCase):
    pass

# Naive Union-Find

Each element starts as its own parent. `find` follows parent pointers to the root. `union` makes one root point to the other.

**Time:** O(n) worst case -- can degenerate into a linked list.

In [ ]:
def make_set_naive(n):
    """Each element is its own parent."""
    return list(range(n))

def find_naive(parent, x):
    """Follow parent pointers to root. O(n) worst case."""
    while parent[x] != x:
        x = parent[x]
    return x

def union_naive(parent, x, y):
    """Make root of x point to root of y."""
    rx, ry = find_naive(parent, x), find_naive(parent, y)
    if rx != ry:
        parent[rx] = ry

def test_naive(self):
    p = make_set_naive(5)  # {0}, {1}, {2}, {3}, {4}
    union_naive(p, 0, 1)
    union_naive(p, 2, 3)
    self.assertEqual(find_naive(p, 0), find_naive(p, 1))
    self.assertNotEqual(find_naive(p, 0), find_naive(p, 2))
    union_naive(p, 1, 3)  # merge {0,1} and {2,3}
    self.assertEqual(find_naive(p, 0), find_naive(p, 3))

UnionFindTests.test_naive = test_naive
unittest.main(argv=['', 'UnionFindTests.test_naive'], verbosity=2, exit=False)

# Optimized: Union by Rank + Path Compression

**Union by rank:** Always attach the shorter tree under the taller one.

**Path compression:** During `find`, make every node on the path point directly to the root.

![Path Compression](images/union-find-path-compression.png)

**Time:** O(α(n)) amortized per operation -- effectively O(1).

In [ ]:
def make_set(n):
    parent = list(range(n))
    rank = [0] * n
    return parent, rank

def find(parent, x):
    """Find with path compression. O(α(n)) amortized."""
    if parent[x] != x:
        parent[x] = find(parent, parent[x])  # path compression
    return parent[x]

def union(parent, rank, x, y):
    """Union by rank. O(α(n)) amortized."""
    rx, ry = find(parent, x), find(parent, y)
    if rx == ry:
        return False  # already in same set
    # attach shorter tree under taller
    if rank[rx] < rank[ry]:
        parent[rx] = ry
    elif rank[rx] > rank[ry]:
        parent[ry] = rx
    else:
        parent[ry] = rx
        rank[rx] += 1
    return True

def connected(parent, x, y):
    """Check if x and y are in the same set."""
    return find(parent, x) == find(parent, y)

def test_optimized(self):
    p, r = make_set(6)
    union(p, r, 0, 1)
    union(p, r, 2, 3)
    union(p, r, 4, 5)
    self.assertTrue(connected(p, 0, 1))
    self.assertFalse(connected(p, 0, 2))
    union(p, r, 1, 3)  # merge {0,1} and {2,3}
    self.assertTrue(connected(p, 0, 3))
    self.assertFalse(connected(p, 0, 5))

UnionFindTests.test_optimized = test_optimized
unittest.main(argv=['', 'UnionFindTests.test_optimized'], verbosity=2, exit=False)

# Application: Cycle Detection in Undirected Graph

For each edge (u, v): if `find(u) == find(v)`, adding this edge creates a cycle.

In [ ]:
def has_cycle(n, edges):
    """Detect cycle in undirected graph using Union-Find. Time: O(E × α(V))"""
    parent, rank = make_set(n)
    for u, v in edges:
        if connected(parent, u, v):
            return True
        union(parent, rank, u, v)
    return False

def test_cycle(self):
    # 0-1-2-0 → cycle
    self.assertTrue(has_cycle(3, [(0, 1), (1, 2), (2, 0)]))
    # 0-1, 0-2 → no cycle (tree)
    self.assertFalse(has_cycle(3, [(0, 1), (0, 2)]))

UnionFindTests.test_cycle = test_cycle
unittest.main(argv=['', 'UnionFindTests.test_cycle'], verbosity=2, exit=False)

# Application: Count Connected Components

In [ ]:
def count_components(n, edges):
    """Count connected components. Time: O(E × α(V))"""
    parent, rank = make_set(n)
    for u, v in edges:
        union(parent, rank, u, v)
    # count distinct roots
    return len(set(find(parent, i) for i in range(n)))

def test_components(self):
    # 0-1-2, 3-4 → 2 components
    self.assertEqual(count_components(5, [(0, 1), (1, 2), (3, 4)]), 2)
    # all connected
    self.assertEqual(count_components(3, [(0, 1), (1, 2)]), 1)
    # no edges
    self.assertEqual(count_components(4, []), 4)

UnionFindTests.test_components = test_components
unittest.main(argv=['', 'UnionFindTests.test_components'], verbosity=2, exit=False)